### Task P2.1
### **Kalman Filter**

In [1]:
import numpy as np
import plotly.graph_objs as go

# Constants
G = 9.81  # Gravity (m/s^2)
DT = 0.01  # Time step
PROC_NOISE = 0.1  # Process noise std dev
OBS_NOISE_X = 0.1  # Observation noise std dev for x
OBS_NOISE_Y = 0.1  # Observation noise std dev for y
INIT_H = 20.0  # Initial height (m)
INIT_V = 30.0  # Initial speed (m/s)
ANGLE = 60.0  # Launch angle (degrees)
SIM_T = 10.0  # Simulation duration (seconds)
NOISE_STD = 0.1  # Observation noise std dev
DROPOUT = 0.05  # Observation dropout probability
VAR_INTERVAL = True  # Variable time intervals

# Simulate Ball Trajectory
def sim_traj(h, v, angle, dt=0.01, t_max=3.0):
    rad = np.radians(angle)
    v_x = v * np.cos(rad)
    v_y = v * np.sin(rad)
    t = np.arange(0, t_max, dt)
    x = v_x * t
    y = h + v_y * t - 0.5 * G * t**2

    ground_hit = np.where(y < 0)[0]
    if len(ground_hit) > 0:
        t = t[:ground_hit[0] + 1]
        x = x[:ground_hit[0] + 1]
        y = y[:ground_hit[0] + 1]

    vx = np.full_like(t, v_x)
    vy = v_y - G * t

    return t, x, y, vx, vy

# Generate Noisy Observations
def noisy_obs(x, y, noise_std=0.1, dropout_rate=0.1, var_time=False, dt=0.01):
    x_obs = x + np.random.normal(0, noise_std, len(x))
    y_obs = y + np.random.normal(0, noise_std, len(y))

    if var_time:
        intervals = np.diff(np.insert(np.cumsum(np.random.exponential(dt, len(x))), 0, 0))
    else:
        intervals = np.full(len(x), dt)

    mask = np.random.rand(len(x)) > dropout_rate
    x_obs = x_obs[mask]
    y_obs = y_obs[mask]
    intervals = intervals[mask]

    return x_obs, y_obs, np.cumsum(intervals)

# Kalman Filter
class KalmanF:
    def __init__(self, dt, proc_noise, obs_noise_x, obs_noise_y, init_state=None, init_cov=None):
        self.dt = dt
        self.state = init_state if init_state is not None else np.matrix([[0], [0], [0], [0]])

        self.A = np.matrix([[1, 0, dt, 0],
                            [0, 1, 0, dt],
                            [0, 0, 1, 0],
                            [0, 0, 0, 1]])

        self.B = np.matrix([[0],
                            [0],
                            [0],
                            [-G * dt]])

        self.H = np.matrix([[1, 0, 0, 0],
                            [0, 1, 0, 0]])

        self.Q = np.matrix([[proc_noise**2, 0, 0, 0],
                            [0, proc_noise**2, 0, 0],
                            [0, 0, proc_noise**2, 0],
                            [0, 0, 0, proc_noise**2]])

        self.R = np.matrix([[obs_noise_x**2, 0],
                            [0, obs_noise_y**2]])

        self.P = init_cov if init_cov is not None else np.eye(self.A.shape[1]) * 1000

    def predict(self):
        self.state = np.dot(self.A, self.state) + self.B
        self.P = np.dot(np.dot(self.A, self.P), self.A.T) + self.Q
        return self.state

    def update(self, z):
        S = np.dot(self.H, np.dot(self.P, self.H.T)) + self.R
        K = np.dot(np.dot(self.P, self.H.T), np.linalg.inv(S))
        y = z - np.dot(self.H, self.state)
        self.state = self.state + np.dot(K, y)
        I = np.eye(self.H.shape[1])
        self.P = (I - np.dot(K, self.H)) * self.P
        return self.state

# Launch parameters
theta_rad = np.radians(ANGLE)
init_vx = INIT_V * np.cos(theta_rad)
init_vy = INIT_V * np.sin(theta_rad)

init_state = np.matrix([[0], [INIT_H], [init_vx], [init_vy]])
init_cov = np.eye(4) * 1000

kf = KalmanF(DT, PROC_NOISE, OBS_NOISE_X, OBS_NOISE_Y, init_state, init_cov)

# Generate synthetic data
t_vals, x_vals, y_vals, vx_vals, vy_vals = sim_traj(INIT_H, INIT_V, ANGLE, DT, SIM_T)
noisy_x, noisy_y, obs_times = noisy_obs(x_vals, y_vals, NOISE_STD, DROPOUT, VAR_INTERVAL, DT)

# Apply Kalman Filter
kf_est = []
for i in range(len(obs_times)):
    kf.predict()
    est = kf.update(np.matrix([[noisy_x[i]], [noisy_y[i]]]))
    est[1] = max(est[1], 0)
    kf_est.append(est)

kf_est = np.array(kf_est).squeeze()
kf_pos = kf_est[:, :2]
kf_vel = kf_est[:, 2:]

# Calculate RMSE for positions
pos_rmse = np.sqrt(np.mean((kf_pos - np.column_stack((noisy_x, noisy_y)))**2))
print(f"RMSE for Position: {pos_rmse}")

# Calculate RMSE for velocities
true_vel = np.column_stack((vx_vals[:len(kf_vel)], vy_vals[:len(kf_vel)]))
vel_rmse = np.sqrt(np.mean((kf_vel - true_vel)**2))
print(f"RMSE for Velocity: {vel_rmse}")

# Plot results
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=x_vals, y=y_vals, mode='lines', name='True Path', line=dict(color='royalblue', width=2)))
fig1.add_trace(go.Scatter(x=noisy_x, y=noisy_y, mode='markers', name='Noisy Obs', marker=dict(color='red', size=5, opacity=0.6)))
fig1.add_trace(go.Scatter(x=kf_pos[:, 0], y=kf_pos[:, 1], mode='lines+markers', name='KF Est', line=dict(color='green', width=2, dash='dot'), marker=dict(color='green', size=5, opacity=0.6)))
fig1.update_layout(title='Kalman Filter: Position Estimates',
                   xaxis_title='Horizontal Position (m)',
                   yaxis_title='Vertical Position (m)',
                   xaxis=dict(tickfont=dict(size=14)),
                   yaxis=dict(tickfont=dict(size=14)),
                   legend=dict(font=dict(size=14)))
fig1.show()

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=t_vals[:len(vx_vals)], y=vx_vals, mode='lines', name='True vx', line=dict(color='royalblue', width=2)))
fig2.add_trace(go.Scatter(x=t_vals[:len(vy_vals)], y=vy_vals, mode='lines', name='True vy', line=dict(color='firebrick', width=2)))
fig2.add_trace(go.Scatter(x=obs_times, y=kf_vel[:, 0], mode='lines+markers', name='Est vx', line=dict(color='green', width=2, dash='dot'), marker=dict(color='green', size=5, opacity=0.6)))
fig2.add_trace(go.Scatter(x=obs_times, y=kf_vel[:, 1], mode='lines+markers', name='Est vy', line=dict(color='purple', width=2, dash='dot'), marker=dict(color='purple', size=5, opacity=0.6)))
fig2.update_layout(title='Kalman Filter: Velocity Estimates vs. Time',
                   xaxis_title='Time (s)',
                   yaxis_title='Velocity (m/s)',
                   xaxis=dict(tickfont=dict(size=14)),
                   yaxis=dict(tickfont=dict(size=14)),
                   legend=dict(font=dict(size=14)))
fig2.show()


RMSE for Position: 0.04815765615739913
RMSE for Velocity: 1.405028804399821
